# QSV2G — Stage 04: Performance, Scaling & Classical Authorization Baselines

**Research purpose:** Complete the software-only performance evaluation for the QSV2G paper. This notebook uses the **exact SHA-256 verified Section IV reference implementation** and standardized **ML-KEM-768 / ML-DSA-65**. It measures (a) end-to-end reference workflow and phases; (b) QSS setup and reconstruction scaling; (c) instrumented classical packet sizes and *analytical* quantum resource use; and (d) controlled authorization microbenchmarks with one-signature and two-signature *illustrative classical baselines*.

**Kaggle instructions:** Import `.ipynb`, choose **Accelerator: None (CPU)**, enable Internet if dependency installation is required, then **Run All**. The final cell creates `qsv2g_stage04_performance_results.zip` in Kaggle Outputs. Download the output ZIP **and** Save Version / downloaded executed notebook to preserve actual Kaggle execution provenance.

**Important distinctions:**
- **B0**: an illustrative *single-authority signature* benchmark, *not* a production single-authority V2G protocol.
- **B1**: two distinct classical ML-DSA signers authenticating the transaction context, but **no QSS step**; this tests the incremental cost of the QSS mechanism, not quantum advantage.
- **B2**: the real QSV2G `threshold_reconstruct` + `issue_token` reference methods, with token-signature checks.
- Only a *common pre-authorized context* is assumed for B0/B1/B2; these microbenchmarks do not compare full protocol security, nor availability, trust, or formal properties. Two signatures authenticate attestations; they are **not independent proof of actual quantum reconstruction**.
- The recorded AEAD packet sizes omit other classical traffic and quantum distribution. The quantum communication counts are a **protocol-model accounting estimate**, not a hardware/network measurement.
- Results from OpenSSL subprocess and native `pqcrypto` backends **must not be pooled as a single performance distribution**. The environment manifest explicitly identifies the current backend.
- The quantum resource is ideal and can be consumed once. No quantum hardware, authenticated physical quantum channels, crash persistence, or deployed V2G energy measurement is evaluated.

## 1. Benchmark design

The notebook runs **5 warm-ups + 30 measured repetitions** for each QSS vector length $L\in\{9,27,81,162\}$, rotating authorized pairs $AB$, $AC$, and $BC$. Separate authorization microbenchmarks use **5 warm-ups + 30 measured repetitions for each pair**, holding cryptographic backend, pre-authorized transaction context, and signature-verification count fixed.

`total_wall_ms` measures actual local `run_demo` runtime, including Python, local cryptography, simulated quantum encoding/reconstruction, and application checks. It **does not include quantum transit delays**. Individual phase durations are drawn from reference code's phase log. Repetitions are not independent physical deployments: results are wall-clock measurements from one runtime.

All loops export raw samples, not just averages. Report units explicitly and use median, p95, sample standard deviation, and sample counts. The fixed NumPy seeds govern only simulation randomness, not cryptographic keys generated from OS entropy.

In [1]:
import sys, os, json, hashlib, base64, zlib, io, time, statistics, platform, zipfile, math, subprocess, shutil
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
import importlib.util
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

IS_KAGGLE=Path('/kaggle/working').is_dir()
OUT=(Path('/kaggle/working')/'qsv2g_stage04' if IS_KAGGLE else Path.cwd()/'qsv2g_stage04_outputs').resolve()
OUT.mkdir(parents=True,exist_ok=True)
ZIP_PATH=(Path('/kaggle/working') if IS_KAGGLE else Path.cwd())/'qsv2g_stage04_performance_results.zip'
SEED=20261027
L_VALUES=[9,27,81,162]
PAIRS=[('A','B'),('A','C'),('B','C')]
WARMUPS=5
REPEATS=30
LOG=[]
def report(*args):
    msg=' '.join(map(str,args));LOG.append(msg);print(msg)
def sha(b):return hashlib.sha256(b).hexdigest()
def version_or_none(name):
    try:return metadata.version(name)
    except metadata.PackageNotFoundError:return None
report('QSV2G Stage04 output path:',OUT,'Kaggle detected:',IS_KAGGLE)
report('Benchmark repetitions:',REPEATS,'warm-ups:',WARMUPS,'seed:',SEED)

QSV2G Stage04 output path: /mnt/data/qsv2g_stage04_work/qsv2g_stage04_outputs Kaggle detected: False
Benchmark repetitions: 30 warm-ups: 5 seed: 20261027


## 2. Cryptographic-backend preflight (never substitute fake PQC)

In [2]:
def openssl_supports_pqc():
    exe=shutil.which('openssl')
    if not exe:return False
    try:
        kem=subprocess.run([exe,'list','-kem-algorithms'],capture_output=True,text=True,timeout=10)
        sig=subprocess.run([exe,'list','-signature-algorithms'],capture_output=True,text=True,timeout=10)
        return 'ML-KEM-768' in kem.stdout and 'ML-DSA-65' in sig.stdout
    except Exception:return False
if importlib.util.find_spec('pqcrypto') is None and not openssl_supports_pqc():
    report('Neither pqcrypto nor OpenSSL 3.5+ PQC found. Installing pqcrypto from pip.')
    proc=subprocess.run([sys.executable,'-m','pip','install','pqcrypto>=0.3.0,<2'],capture_output=True,text=True,timeout=240)
    if proc.returncode:
        raise RuntimeError('Install pqcrypto failed. Enable Kaggle Internet and rerun the notebook. Last output: '+proc.stderr[-1000:])
    importlib.invalidate_caches()
assert importlib.util.find_spec('pqcrypto') is not None or openssl_supports_pqc(),'Genuine standardized PQC required'
report('Cryptographic backend preflight completed.')

Cryptographic backend preflight completed.


## 3. Verify and extract immutable QSV2G reference sources

In [3]:
CORE_B64='eNrdPf1z27aSv/uv4LXzhqQNybbc5uWUshPXUVtf81XbycvVo+NQJCwxokiGpGy5ef7fb3fxQfBDtpt05mYubWKJABaLxX5jQX/zzTe/n78f/WKd87CKs9Q6fW8V/IoXPA25lRdZlVW3Obecgpc8KMKFVcardRJgX2a9fnNhRTxPslseWSUP1wW3ytuy4it3uHOerYuQjy2C76+COH3qy1lO3/tn/DoueTSs+Ga48/s6SKv1amzxTRBW1uifL6xqUXA++LSuiriyyiqo+DWMzYrSuokrwCJY5QlMmgdxYa14UMLUK55W5XDn7e8nYwvArIPEevVy8Nvk1eCfT55a+/jlxfnx4Mn31nUcWPmnsLjNq8x6c2a9yXl6fv7S+tGzjobfD3cuFnFpwf9pVlmrIInDOFuXA7nAa17EV3EwSzgs7XxoHa+rBcwch4BjZIWLIE15UjJFD54WWZIgbmznk1gnILzKilsrSCOruskGAUDIYJ231iJLeQk/AySkJrQVlOV6leNHWN8333yzc1VkK8v3r9YVTOH7VrzKs6ICgIAxDSl35KNZUPIn36lvi6BcJPFMf10Fofr8scxS9XkVVAv1OdOgYEEFr+qvi3UVJ/rbegbcEvJStwMX5FdxwvX3eMUF4lFQBWECi+Klwlw/YtZVzJNIdMwBD0BXdXqLaFED8GScztXz4/RWTZICnYB8sHW56Ck2eV4E+eJ2uAj+hKUN8yJexVV8zcthGOcLXpTDgAeRBjc5/+Xk1c7Oy+OfJi8tz7KJgwdnk58nZ5PXJ5PB6GD0xN45fnfx65uz04vTyTl0cuxjm9k/wd8T2905fwmS4Vmf4eH4AB+PD7FhPLrb2aFlWm9RtMIsOZ7BlM5kE3LaX3e8Y8GfHLrs7ET8CoTx0zouuBNmaRQLqeObPAnSwOgeXxGr6j7iKf4pApCy1mTmeDEJ7r2fzT46WQ0wLuMU5A7UgJMxa3Zb8dIdAzrAcSkszPdRL/i+PbapzWaWLXjNHoufw9mT70CLZBEAcIcRp092UIZxbLt3/dNUa5DqLdNQG04TA2eV9vhSY71xraussDZWnFrZdAvoJC6rGvK2wf1jozg0xn4uq8JZumMN4lqAWLJrBFICjXnkZEPC03G3rdUBMCwGtTDLssR1LYCQodJ5DTpAz5Xt1Lt4AZSYFEVWOFf2u7Rc5znNJATG+oyEQkr7fhqsgGR3Y+sqyWCb0/kgz2Ai6zpI1iByqFzgK+qsLA2S5BbWV8LP7IZHtuSIMEizFGgj2EEigwseRiBhpWOQj9GK/SW/Lb2LYs1R8+VBEaCy9hwQCAabB714ikraJwbwfg6SkrtDyR/2uroaPFVzw+CS+wKDIriROAhhKAEq13h16brpbLQacys26Ra3aDPdPlzstW6XfTbDOa+cmh1dz5N8T1q8hMYNPDMYlilxuGsCM+hZy4mUjs2lGjRlsFkxaEVONHUfg48QkO34CLnZjg6Nd3rpdSkHT5uIKIFYjtWopihslAzc7bSGbEy+UoOJv4BloxL3XasMyRyuZA/JM7vE9qXcKjCKP8cbEIasiMA+p3PYSF7xYhWnwA5xyNBk8GiQ8HReLQaSQ6E/QYP+QzSrCClbVx7ubFAUwa0j1otLusIFiTlrCoaLdbr0BKte1bQBGNaeZ8FkDvVwh1XmE7s43wFfxHPb3aMGkwqiHYbKdf5aL1F1keZ7WC6C0fdPnBYlQMnGcyAlII3jwbY7S2Z1gUDDMOU32NiCAMpVgLZbsMLK55+cgM1aYMJsBYTkvuyMHaj/Yhld+XxTFeCHOaBbKmbFy1UPDrpt69QSVA5s7eQF4BynV6A9U+9oJPeedsy2n1n0Q29YjBtWBOmcO4cMPZphyOPESfdh4N6hIeKVp9GhCao9nGJP7MdlPO0lyzOcds+rzB2EJ5fjdCq2T3T0ARMnCWY8gZ0gYaiJ8GurYWfnW2sAfyzpDjfdXmzZOR15aT7kt9wZsQg5Gr/iJiR8czh66u58oA6Cdy8vD9jhlF0esoPplMle7s4f1CWKg7kDTYPDqTsMSjIeusuvDSiHEgp0rcHsQ4fyE3gTI3fnBOKA5oAD/A8GAQL6E/53KD+1UAI/66UB4FD17ZtMEBeMCfcF9/pST4An7c/iSmkE4EVPPaIHRE2cZVmAtMLPJAbrNx8CaxTxxs/B/BXOHyMWuM+3NH4YAX+z05H7HBEWjIbdNNBfR9SK9HhO0wlnrshmwSxOwDXjJS1zVjo00N3dHQnnINp4YJIdosF8FWycxiDXlWwmfEHoFZdhkpW82e0SwEyBtmB6E++QDw5HLviePEksIhF6zFdBnIAVtl2Tbx0Yt78/YvDjH00KgwCnJboZfgXsAn+1QxBmEAfxqKMqkdyebQ8/gscBfspnbB8fPJ3d2SSU+BXlEkFp4TWEdlkL7QFD/Uk7ykaGuAawszOvjwMcJCEOuFyC0NZf9oDH3YZ23vPQhwsCdw9/zmZuc1VDUF0c9A2CQKH+OP6493QKWBCOH9s4os5mT123q801RKDqm1eTX45JfDe5M/q4C5/yeP8IZPYIn2KU6EjhPpLCrfgf93QTl94BSO9RLb0EcXf3YxurIxfstNjGT2XpEyc6hqX8d17G8P1HiFAgrvQL698FK/ZK+Dsqf7T2SdKOXG0RFduVOIODwot8dZqSh2JhgC7ZSUvYn7zISgeWAf+5rbXorS5MhMdi8GXBHMDE/ccR/hzt4qepd6jl/6hBYyFiMkyK1iGPfGQLCKQdWCGzlhBTyVWvYBsiRA0/IC2pB3ZgB+4QHJBFkCPdB4eNGWjYc/oX1Gz60XGHF2JCmXPwMQEhZsNPzCrQ9dj4V8E6qaSXq+l+TkkL6yM540iPtQjWg8RMYVjZFWYFIL4Gy1paFMUG1lFPLmRIgOVkVpDnSYzuvSV6Dj5YHKMFq8qsasGt88nJm9cvwDVMYDDmaxC+NeOwF1zOj36QQpZ+rtm1d4nB7GU+pU3LcdNwpdPmjjfI6sAwlyGnSJlALQWxgFR9NAhVnzsE9nOIs0f1JqgtUDoRKDoMF1kM7vl/stwjWPv0Lw1Xkr1Zs821F8XXqyxChcYkr6CzLCg0tqCDs7kG4/+PI3OXaajiIxJYv7xdzbLEoT2lZh2FpNnKcwRJoPESbOKg/gZWrQXbgQkHm7W7CybEodHAY4AbdtMSCqxVgpsK2gzUqJjoQ2CaKvr54Yh9AHH6IBf2R7fHH0fsD+jxh+wBAlp6l1MtcWVD4rQ+hC3zTD1hNvhXSVB58KGzPWqCIfAd6srPjcjAhgC9ZM1HaVas7DHFpIjxdZRVjpqCqQ/IB2AYW0M3iByaOdAsPnG1AUiaagTvfAieK0gDDbIN7c/HQvvjMdAgXPdJlsALFmEkJid64fVpqZk7UFrfBWsgTMysoczrCe8a2gnJr53GY5HxRAyioIhgbZGVZ2U1UElHMxMG/jMomDhBdXMFof8sCJfSxZTpqd9PqHsde/s+hFKV7zugQK4M7sGvw2WcRh4mLmqnurhtBpoioSeTrsMlX6l02yrx4Zv/zydP7+lfxvPUGBCVgf/k+0Z/gQdfeTW8Z+IhjPXuG4O42zodjImxa25mjvfqzLGth3PK2FnOKeFEaRn2KovWCX+dVT9n6zSiZ61MAt9wT+RNhzeLOFw4dgbCU5aJ3YnwoWs3UgefovLqVOuwWKfOJfRkNmY9IMYfwNIHwG+YTl5gwM7CIKckMTgpYG9EhqYC/6Ynr0AkAWLdOwHuRIAQv2oaWKBdk9gmlxDWNiyrCINnzGHYNdUprRbPZWuXLK29VOn8o+H3e1s20nLqRbr2doByezwgQXuD9IQ6addNrFDe7mydYu5bpO7sM944l9ivUVqnwTW453iyMLQmKZ0w/BbM5/DjNK14kXJBmDxGKwyCDo69FhDAwCLRjUPUAutZWcXVGpztmxi6zbi1xtMWyWYkz3JpPvRPSKiZBbHHmk5RMMqGbfQxf1+SYDNL7Kx8BGa3Xi6dyKh0//CCo0gExe0LcBrRTQE9V/CreOPZn8rr0dwHnH3bxST9VZZEvGjSrciyysNMvyNam7yDyhHTmwwPDMgfNRDFFOrnO1clncaWg8D2sb87vAFW5TIJg4ObcGHl4OmYWz7du8TYgCA415eHY4ghWtnCawY9XNqTa+DNoKhKJIVjP4flcXD9LIQgU2KUENMEnjZmz9sih+gwS7C72fb29O2EnoPV6j4PbyIP8WV01oJDjw46egUiDbIeGDE9jmOVOBkGJA4tZBsKJ3k0tuy9fCiwamXtGJlXiAgLnidBCIHn5fj7g4P+PKKTSwln1u6loPyG/IFI7puZqje4cVpzNWr+Jb+dQ1TWslOmxDY2S4mQ3dLV+XrG8iIGl1GZlqGETHywCMqggu1VjcwWrXrr1SB4xjHXiHhRtOC6z9SK1Rx64m9hBbeWHEJHfyjw61kShwNskTk1bBH+O4gmiR5JnBUpkftS2UT6CdmMmtQol0IoI3ffLpfDnK/sZ1ZuPMzFw+agliEx5YvZsMoc1oRGRZsS+GKYBWiBfbZRzJxyCY5QuODhUhiXHoPj/oXJ1cxxqsHDN6A0TWjMm//VedXWLhucC1LbfKBZFrz+IC+FBob5v4ZlVVysWE+CRqj9LCs6dFgWHiN8Gqgn8FlYsbL0qFfTelwSOddVgpQjmLYgJlIXaSyI/TxfatLaz0PhTtChMn4v0Y2wPgMoewxD75hjUxd4biRv5BIJFU1C0Dk1CUGcWOMI56vJKMEryP2kFJ06pITHgpRysEHOR9CSgDZIWC4V126h4GcQTyAgTof0G4cVUhKaWJeKBgnBvfq7FKdJPvTatmpNaOzVmjioqzW/VKXhyv7vVZr0ZP8/abRtCg0jBEMWV+X87+Qmgq4Bm+KEkcsD8oSDt4oTwDPUE3RtSRO2j+Efkido7ROoeK7JQHVCt1qvI7qI49fQohNUi6lLzDVqAsl5zSm3nfiKoSp+Qfe5foJs0IpdKeLVpSr6HIuynOi6yJTwuYoQyX9sBvLW4XDDrPdYiCBawZEhNx/JVVLh0tbFPrS5YuH3WZ7GPgNhUIGYey1NT73VjDZ6DP/c9YpEg0iSQKbz3KTRzs5zXeck8ywXRXB1FYdioYDrmIRJ5FZ5GvlhtsbQbxynlXcgH/N0XPLKozNbBzgNc5v+VUC6D9igargWZHwED1YCOAuisSznMdhXpviJiYyJfxjt7v7nk8Ehs48n54NfTl5ZsgFWuwjWJTKCkbtIM4iLvA6U+vj7cCTPv1Vuxui15x3WB+uVJ8qwlIW9dbVXQpPAcmAhHQH8bFOrPRadbFHfhYkIsoSmx2BQZkmwxq1lwONLCW7aoRSe+1Cb63mHI3LT6SsVYsVKX8FegWewzhOqDkQBWwUJxDArHlnHk+MXYoxBQGR6K+9ZvEJYLh5RM9Y2bdCiLalo+7CkqKcWzCYsgrqEEbvrQ0Ihu7yZ68NFDYMokstv70Be9bD57+fnZ7wUBaE0IF2PwRTJ7QgSzLiL6sIx1t/UZwvG9zBbreKKoudWJwTl2e9evzufvBDWOsnC5ZjqWf6Nmq1OTcoqIl5cC2ZnQcjQvdgqC3TAoYEzG1aCAGgpVpCgAbxVELH6BIvwSojxI9vt5RmszKEJgW8E26Aeo1SGWVCIcW6gzlrAjlOdayz0qwHZwNE+m5xPzt4DklK0kAaeA+sTdT1iUoMIiGhVrMOqQQiGB1hVsMp50Ti+2koZPa0sPFIzd6bWtANSCT5bxeUqqMJFg1TqYPlzML6c1nQwiHN3T5IZugt+ao3pmk1RppSXVCbCU9iygo4/9NrKSwFp6vYnHMWBU+ccUBNRHQI6gppEHgERqHYsyAVkOnDdXvCaEgoNddCy/YjK/VJa8E1OZ4Ler44sdoWtGpy8eWUzAYERWdI162DV51vIilWqGVKgBQRDhjUAxReaH7GKWvfrYxKpDcDGA8qXeAhYoxVML5f9bOP+46hdYoACWQ+FTZm67rTPzqvZdBuVbyR9DoqUxpM3r8/fvVL6qNujHIYJDyCsQcdJlnrDwuOi4OQJYfLXPBTet4A/8Zs4NUz5jY9ndkoVOS+9wycj5DvSdapMA77CX1RLWGEAHJdhRCo8BmhwsJLZlQrZkxoYjPWSp/6Cb8Bauz0qmkRTdUagM55kN86RyM75NXlfuvcJsCADATNPH+uTyuassLP3QRMcQ+C6TBwwYN0OOPcecHLfDbvldEAwsQImp3Z3tLGDyetD/zidJxyi9LKkrF1VgO0y7ljEWBOAWyx0IgW3WKEdh3EFCg5cLGCL+oZAfStA12d0TueY8LVl3h7sdeThRjPBI/qQweQUzZzgqHvSVQfiqGNAR/lryFH9rITTtAxTPRNaPf0N4x7ji7hmQQEIpQBMkW3Ce+m91LZt7l1OFU6YWr71+TVN4zafhmXjKd/wcI2ajh42wd+A1hJuqvf5DsXyFexZMMebGH/iTYU0TNYRt2q3a78K5qTGwUOewW6he0xO0TN0wagz7m6Mzt7xi2FztoLPBcqm0Akn+WhUL2IuVtDbqbVzceSvAmCX4gGg0A8mtvm1H9OxUpD4mh39g4NDuwk3LOMI9Nm5/2Ly6o0Pzc/U0Rg8/fX47JdJawBES1QMDqKoWGpopJfcB4y6XLiQOq+r6JhixNb6qwMqF8HYa4j/UJ5dKdw5qOTXk38p3H0Q1AqoVSw96g/2+UqFIUYmAzvIEGERlChLIKXr0rPfHp+f28za3Y14BYxcNmKHm60wNa7Av7puwibYEHjSFLaYAmJO+slsngR5yWFr4VmB58oOON43g9Yy3N3Dg4MDrLbSON215jTWDAAMR3COpSdFO+2nzPhnebHl7gevKdTMfncyOCb+hw8/qQERbW+WJmAvJXBxlGF6ece9LHr4xIhhrrHk0MMiZkNcGLibv9isZmNWHLvG4cnR6CcsLbQGP1qHo6cW1j4OyHVWZd4lE7WQgCZp2tHg0xpFV6rVrNjpd2P6yiEFii6zHoMkuDmT94SQvEAWtomihLhecy3RJrAtlGNSCPaePvnu4MC9HB8+mQ7RiBtk/+lBsodll+whcOHJeb0mVAis+Mn9C6QSYF32KLguPn6YVFKqYfzp+cXkrA58SGrtBusxC3yUhJz00s+x6gei3erWo5ppJQlGFCyDQ0u47qguKRbSD0AnPxQczUVwpJFjtpI0cB6LsuqJD79K1PKgANTjPBAXffDuSA/RUr6NBSTxUj98kEkEp5W+zPRUB8/0o1A9anbeJs2T9z7aAMkAwO8GJ0gjI0MPzvSsBiLrGXhewpzrWcz0qt6/sdFXFvbrB5cH0/85nLp79YPDcTfhg5WLbfsyqHH6wTsC/YvaG3wakHTsUeL0PfvsNFcLTlgja6T9GVIZIqskntnbZE6jzv4mEgt1lRdZhhbiI4VwbV7SeFI2qLuox2gVE7capXwbbsCdGsnQ7dnpsByr2eQ2i29qj+W3v7zBYXODQT89vMH1egDt3i0WNHjcFgvM2d9NQ6FsH7XRYWlsdHNx9YB1gNWsV11RBzwhJv758XzYwJM1T6WapJFz9hjiv2NSIQoh3a2WhzrkB1JitcKqtfvME+7N5PXF6cnxRdtChUUGji3EUUGc+s0UrFGnhfBnSVwufBlCbnPUGoanOS2zTdvWsT98yaKlp311o4am7jPDyjp5XexLiMkNsvKlUWOMnNflFqwAmby3WevGG2JR38Haxg8Ek30hTJc6ypobqsbBfTfpZ8oH1SGIEE1ST9ZcmGu8+PsI1yghIB8rLOK88nFNXmthF8ZNOkMT91Ac9QhS5wSCmhbQe9RRD5nvAyQJW4fPtRR1jx8aRN6qU2gqku/WZMw+PjmZvL2wH1IZXwJSLqTOySAH4eWqnlQlMAcF+ZI5ZCVJ1NzHli0vyctHrpIzUaVXpN6/0DuNiLuNBO75KYj9RW8YgHcu27wi2W9Pa/Qu6+TFEjy+7s1OwhfcR2wP+9tD41KBqNK7pauWdP/RuBXavupJ7eqSuLvX3hEzBQK4jUJPAgahkWSYjE7wrBEwG3HdGqrWk9Gkh0MIlPUfnhjFbFFYIu7pVOK8FldQWutUZbeTWyuOBAubpdwA6VqwroHYbyeohQRSZbM1pFYQnhrEag1Ep6u8Chaao8HPp69Pz39F1a62/rGCiwCJ7ZuTIkqPhSwsIqpGxCcuVlvkFqYKJe5ypWiRHzEDQyTvWUDYWADXVPtL4IXn8/AqRGgBzCXP6gV/qHgJ+apugG/9bsD55Pz89M3rpgMgDbqvTTymgTFWDUIIvCLDIoOCYVTbWmtPmamkO+YVlhfGkfGIPg3x+BZlXxHCvAX5rTUBUHjqlRARYPng28AXnQsUb4pA64f35PAQWVU+o99TxLM1ztEIn0Ef/klkxN1GLGUcbeQK8QUNEKv4q5sFhN+YtIIHVxzdaUyse0/wifLm9WofF24rKjNFWsJek7f3UDajK22ORs9+cXouP4vSbo3uj+D+yyNYWJAq0M0wucSL+a3o2GYcWHmWt5J/2M/7bOO1SHtsT15Pzn75b5h4FWxsLPRw9IzAousUYI7t1b8WditFep1EsjS5OhiMdPpHAKgJ6rYQCquNt9UR+cseSb0gppAy5B/nUq8YUJPXzYF2KJU6vvhgt/BA1AwTtJR1TiBnuhwEZ2mUP7SS+Ze25EU/uwLOtqckIGZxh7unn5g1Fb3xpW5vFYu3B6vws9OgItFugxmUgq4Ho8Pja5XbwBXrWtJlu/alceJaD2VIGxm/CBrQq17mBacTpx5pUJsIWP7QSafLtBQ2H06ZIiueVmEyqs35QVh7I8cntsClxYtZWgIeXccOw5c/0A87hyBG8kRgBPygIEGAgDfsseYXOwjltyAk5BC0fCK/3ZkAfM1NoM41N4l3eUCzyzrMCcv4rWZQ9xEcB3Akv8kJWzynn/bzXcsxVLgqPpCjH0aU1atqcgM82WLwpN2CLbiYfLhoBa5qdZqRmIIoAhG5C8LvdNsWorr1zdpIvLhcZHRXtfESLQbLv86WIAAy9zoLIl9xzKPMgUK+ZlVwnUvw1L8m9yrTrXWKtc7GOlfrJJEvwBNRYzGLK6xNHmQpx8JSPE2ssgRsR1q5Xyd+lt0wuD1SqGhlSoB6VZhBS3HqqpKiRl+dFzWf9WXOZEQoerDHCrMrKauyTpbgy7JfMSHSkh+Ig+XnrQFeQ/V0bJCrsWjwRm9YBdIWU2UqHmiKSi3NsaKAS6f1NNM86owzCLE+D0tYgt7uuj5YY4A1EFuqygx47o/eiNlX/IYXVrUIUnqbAN5NjtOwqnHflrB6+/bszft2rsoQ3fpsQYkuLAFkR7zyo4lJLfv4PsdykSWR36k8w1MZLfquLD4TV6ofJ+caZVbjqdd5j8yT1cfaNFWKJ+/w/4cnr/brd2lRrx+oaKC5QGaTnKeaEgahNSo4epuwq3PtbqWhrjJcrcuKbnCmeIezc2KI1DPq7PrP0Ydm0SMaSD22r/auPbS5YY3h+jReFQ2a29dBRgPCEIFMhZbTs8mJ1g2siYBZd7btEBDVy8XZu5NOlhWLBzTz2YLZ1PtpRHFlvW7x+jpVwen1bFDN0MBVa+7TEZlgY7VwVf3+2BNCE/NOLZwsk2wxML6WQ+9ATRBEEM2xp3Ufat4WVbVrbZIXALJtW6Qd+56k2qqc1w6+mt7YIkWLTjFI6pilIli1ByYNL3TUmlAv8K7HD9eQWxV4esLLtXbBGw+1UWs8NK3at8LOm0fPzyy6a61TwiIylsVZ+zOsysC3rNQSL5JB+E4mcXVd12x1T53k5WsijOEWmaQ5mKqbHArlRvFaLcp1fGoWjMlRbcVBrItetNo3UXNCH+neg5yMrj/Iz3e90nfx5rfJ6157IZMBNJXNCBcMorAoBmxUjZvMXGAp5yrIUTEbqx2K91Zibr4WP3w1MbkNWBmUo9vQkEL1sHVuL060g6uKsEgSujD2OEEVi0R9bynt30x5iKMMs5a/h8Ca8JfG42mT3Fjc7jR61o1Tt08aepfVlQyqDJO8AkzVbvd6prsU45p97+lnyJwhaS2vscy7wVuuv2Jawg7r5rAT3LVKju0sl21ZDvvD5ZcUXPdUjcL8hZ3EK/lVZCzacEDHyXbSdsrtssdkF4inJSuP6cfd9lTG2eT87RfkMuRLT4FC7v0pjaZ8KW7/utyGgvJQZqObuRgPDqd7WxMe0IiMYPoZIrFhvuh1W5KjkbB8r8Iu0qsRxwI6EKDkVlxUUYAtpJSIaJiVYj21eotWkuHbMfDAOwlicvu7GllBuSQWnXrqZl1E3mDdmhuteac1NAcLFpZG4XZfJSd77z+0kQDmVmCyvDVJynUbhKSttrBuQ+YnftgHEI+aFUVFDSdhaUFHUVHt8JlaReBV9wGRnRpxFiXh5bsL7om5viQT1RsJW0Fl4anGPavUEj5VNx5lbCn5KJDBqaUFzLyFXEj3zzOWXKuIaY9fVo9pqP/u2lUbrM5T2RQRjOiWUZsz66Yj1WS6eKSuudHtvk341rpYUBmTLHsAqQnphgKmu5dpdmMtMI57ZsXYlmJS5FbVRuC75uRcRnlEV86MF6nUqD+ZMnE3UaTe0bc22iBQOxq1FvvPNhvWTU+nGBwKn7Vn/RCl1e9da++sCEprWN9NWf3l++n2y1169GNveNUDaveNeETa7PLx/vYXOJUNljQNesPFNHHE4jaNHnXvuZjWrITSJrAntGsWRqn6fFAXNIe6Yaeeb0laYJEb7XM7+sOSQHKOGpWkVARMku81YxoquBFTydwEvvjVv6GTKqy8zdKo9I4OWBWHS7/0vn/IazQRI9JFcSRSg+RDCkqaPmOsDnXupVln0kqC7dKxACsZVFZbRWJzT9WSGkeVXgCUricFVbaCwKcJQkTJ4n7pChgSfxvH7Fb9agt6oXddE795sKYUFtJyoc4vjs8u2j5UX25AuX7ppse7MnP7MAe7d477UvqaZD7d1P8SV+uBdH43ld9BjwEIl4oQ8T148npKT9nelxhREmGMGJX9lK8G7eOVb+mNZHNwrIAximzwr4WFB5UQF19nwImt3ztQ0sGyYhx8ywBGzFgxE6xm8XwNk9bmQRym+uubhXfwTB6+4pfaFbmkk9Ip3XgAYa94Xnp1vXGSzbEKw8dFdoqRWxfU6reCS9HeF3Lttq58RpW3ilNHNCo1MKCZd+WI9mu2ouoH7wDXauJj/WgZJB9bM1Buy+ZIYFXhK4i7HlL/7EbV/tGTg4PdQx8WjX+bE9ZE2yNMNRimCTio+7TeKoLL2PMOnzVQ3fOiqvNKQg3hR0/Dba9CsV7d+Ye6M75xiM7KkejIAHhVnUc99RZiNBYJ1ID2ccf79f+Hycm7rvIXI2sGZpYBtzUPvjQtXJYekYMqFqReMXLY9PsE6tsBdH2qWFMO/1E5BI2mTCPQmybpzjheAqRfR9CnmOWlo9PXF5Ozs3dkS+hNdPX04gzHPnnz6u3LyYV56xQTekX0wGFpupZfGg6jGVZ3bvl27ylBKBb5QSVKGEw9wyx7hb+LAZrX/1qIdsHdrS0giXbde8JqIMCrrwirkRQPBNZqjykq+FtUfE9QyxoIYWxUzyp3bGsxtsq5ASVOX3eLik38meQc8aPzpgYxkXzv8Tr1IzDdTuPqaOtkpnOVdHeX7n+qX1rk0SYpEOIHozH4D4x5aVw+pZHit5joh+pFX417/fjCQnlFzdjmYeO2Tn1PxwCrH9pMSCerb+90u4Wl6mbO0q2+buPQLXkC1WGAz/D9XaqgiNW1T0YX/dBmVBHlst43Aph/GiVTDVD1c5s9QVjdWqqexQvU+yjQObSvj+vNLdSnisy8v8vUEb7RVZ3cqk0xj/WNbsbjPrT6zxOJYelYqnGK2F0wPe+Dax7rdE50euDo7HoPrJ4cdTs93YWoUw0tpu1PXXfH9/brZW0Z5zSoqi1c970uO8arZRovkRGvlzGgkCbKhabCy5D/NRGHW+AnlBBy4TvIuNszwf8CWW1HIA=='
TEX_B64='...'
OLD_TIMINGS_B64='eNqVkD1uwzAMRvecwntpQ5Qokh6dn6nO1AMYaprBiJu4jpAgt68t5wL6Ni4P7zFOfRhgDP0E8RbD0D3DMHS/d/gOp8v5+gNtF6c+3jcGmi2wrTw7NWKWOXYexr/T9BrjrbiG2D/OxbEtPw/HUliLj+XYfzUle0C2G4RmtyCQFZHrZao5AAvbBLDOMq4OTJrl4FIGVkjKqAlBlnMI9K4gNLJWiJDkEHzKwKp2QmZ18FznEPhdYWcFkfTIOjNDUsYsoeIoZSh5yiH8AxOylwU='
OLD_PACKETS_B64='eNo9zDEOgzAMRuHdZ/EUKC2nidzoh0YVcWUbAT09FUPHb3hvgbvM4K0a8vMIOMkaL7X6lajassE/2hw83vuBirbAHlnKm1M3/q3TBOPukQg7ynqVHmLBabhRwJbart/PPZ0dwCoo'
CORE_SHA='2c86ca9eecbc5925100722119f061e640df03f993e14909b3078b23406330758'
TEX_SHA='4ca43b1f6debd2e2bf314ff9adf74f0ad08ba1702e1055eca1943b9533290ef2'
OLD_TIMINGS_SHA='268d6131673c26c237dc1d85588a74974e11b13143cb442cdbbcf1f51f3f8d7d'
OLD_PACKETS_SHA='9b1c4e37dc85130be4041e9de95845834f6f4208ac6d6f5a8ceb1bb7937d634c'
core=zlib.decompress(base64.b64decode(CORE_B64))
tex=zlib.decompress(base64.b64decode(TEX_B64))
oldtimings=zlib.decompress(base64.b64decode(OLD_TIMINGS_B64))
oldpackets=zlib.decompress(base64.b64decode(OLD_PACKETS_B64))
assert sha(core)==CORE_SHA and sha(tex)==TEX_SHA
assert sha(oldtimings)==OLD_TIMINGS_SHA and sha(oldpackets)==OLD_PACKETS_SHA
(OUT/'qsv2g_core_reference.py').write_bytes(core)
(OUT/'QSV2G_SectionIV_reference.tex').write_bytes(tex)
(OUT/'legacy_end_to_end_pqcrypto_eight_runs.csv').write_bytes(oldtimings)
(OUT/'legacy_aead_packet_sizes.csv').write_bytes(oldpackets)
exec(compile(core.decode('utf8'),'qsv2g_core_reference.py','exec'),globals())
pqc=PQCrypto()
assert 'ML-KEM-768' in pqc.kind and 'ML-DSA-65' in pqc.kind
report('Verified core SHA-256:',CORE_SHA)
report('Verified protocol section SHA-256:',TEX_SHA)
report('Genuine cryptography provider:',pqc.kind)

Verified core SHA-256: 2c86ca9eecbc5925100722119f061e640df03f993e14909b3078b23406330758
Verified protocol section SHA-256: 4ca43b1f6debd2e2bf314ff9adf74f0ad08ba1702e1055eca1943b9533290ef2
Genuine cryptography provider: OpenSSL 3.5+ ML-KEM-768 + ML-DSA-65 (subprocess)


## 4. Self-tests and benchmark sanity checks

In [4]:
# Known working auth flow; self-test before collecting any timing.
probe=run_demo(crypto=pqc,pair=('A','B'),seed=SEED,L=9)
assert probe.stage=='TERMINATED', probe.log
assert probe.resource.state=='CONSUMED'
assert sum(probe.wire_bytes.values())>0 and len(probe.token['signatures'])==2
assert set(probe.wire_bytes)=={'context_offer','context_ack','authorization_response','execution_start','termination'}
assert all(z['status']=='PASS' for z in probe.log if z['phase']!='termination')
assert probe.log[-1]['status']=='COMPLETED'
report('Reference full-transaction smoke test: PASS; measured AEAD packets:',probe.wire_bytes)

Reference full-transaction smoke test: PASS; measured AEAD packets: {'context_offer': 382, 'context_ack': 239, 'authorization_response': 9746, 'execution_start': 265, 'termination': 264}


## 5. Full original QSV2G workflow: end-to-end latency and phases

In [5]:
full_rows=[];phase_rows=[];packet_rows=[]
for L in L_VALUES:
    for k in range(WARMUPS+REPEATS):
        pair=PAIRS[k%len(PAIRS)]
        measured=k>=WARMUPS
        t0=time.perf_counter_ns()
        p=run_demo(crypto=pqc,pair=pair,seed=SEED+10000*L+k,L=L,op=['CHARGE','DISCHARGE'][k%2])
        wall_ms=(time.perf_counter_ns()-t0)/1e6
        assert p.stage=='TERMINATED' and p.resource.state=='CONSUMED',('Unexpected transaction failure',L,k,p.log)
        assert sum('PASS'==z['status'] for z in p.log)==9, ('Unexpected phase status',p.log)
        if not measured:continue
        trial=k-WARMUPS
        full_rows.append(dict(L_trits=L,trial=trial,pair=''.join(pair),op=p.op,total_wall_ms=wall_ms,backend=pqc.kind,resource_consumed=p.resource.state))
        for rec in p.log:
            phase_rows.append(dict(L_trits=L,trial=trial,pair=''.join(pair),op=p.op,phase=rec['phase'],phase_elapsed_ms=rec['elapsed_ms'],phase_status=rec['status'],backend=pqc.kind))
        for typ,length in p.wire_bytes.items():
            packet_rows.append(dict(L_trits=L,trial=trial,pair=''.join(pair),message=typ,wire_bytes=length,coverage='instrumented selected AEAD packet only',backend=pqc.kind))
    report('Full workflow complete: L=',L,'measured=',REPEATS)
full_df=pd.DataFrame(full_rows);phase_df=pd.DataFrame(phase_rows);packet_df=pd.DataFrame(packet_rows)
assert len(full_df)==len(L_VALUES)*REPEATS and len(phase_df)==10*len(full_df)
assert len(packet_df)==5*len(full_df)
assert full_df.groupby('L_trits').size().to_dict()=={x:REPEATS for x in L_VALUES}
assert set(full_df.op)=={'CHARGE','DISCHARGE'}
full_df.to_csv(OUT/'end_to_end_raw.csv',index=False)
phase_df.to_csv(OUT/'phase_latency_raw.csv',index=False)
packet_df.to_csv(OUT/'instrumented_aead_packets_raw.csv',index=False)
report('End-to-end measured transactions:',len(full_df),'phase observations:',len(phase_df))

Full workflow complete: L= 9 measured= 30


Full workflow complete: L= 27 measured= 30


Full workflow complete: L= 81 measured= 30


Full workflow complete: L= 162 measured= 30
End-to-end measured transactions: 120 phase observations: 1200


## 6. QSS resource preparation, reconstruction and memory scaling

In [6]:
qss_rows=[]
for L in L_VALUES:
    for k in range(WARMUPS+REPEATS):
        pair=PAIRS[k%len(PAIRS)]
        rng=np.random.default_rng(SEED+50000*L+k)
        t0=time.perf_counter_ns()
        res=new_qss_resource(L=L,rng=rng)
        preparation_ms=(time.perf_counter_ns()-t0)/1e6
        bytes_state_arrays=sum(state.nbytes for arr in res.states.values() for state in arr)
        assert bytes_state_arrays==3*L*27*16
        ctx=H('QSV2G-STAGE04-CONTEXT',L,k)
        res.reserve(ctx,pair)
        t1=time.perf_counter_ns()
        recovered=res.reconstruct(ctx,pair,rng,tamper=False)
        reconstruction_ms=(time.perf_counter_ns()-t1)/1e6
        assert len(recovered)==L and res.state=='CONSUMED' and not res.states
        if k>=WARMUPS:
            qss_rows.append(dict(L_trits=L,trial=k-WARMUPS,pair=''.join(pair),resource_preparation_ms=preparation_ms,
             reconstruction_ms=reconstruction_ms,simulated_state_array_bytes=bytes_state_arrays,
             qss_qutrit_subsystems=9*L,qss_qutrit_remote_transmissions=6*L,
             bell_registration_qubit_transmissions=512,
             reporting_note='Counts are analytical; no quantum-network transport simulated'))
    report('Resource scaling complete: L=',L)
qss_df=pd.DataFrame(qss_rows)
assert len(qss_df)==len(L_VALUES)*REPEATS
assert all(qss_df.qss_qutrit_subsystems==qss_df.L_trits*9)
assert all(qss_df.qss_qutrit_remote_transmissions==qss_df.L_trits*6)
qss_df.to_csv(OUT/'qss_resource_scaling_raw.csv',index=False)

Resource scaling complete: L= 9
Resource scaling complete: L= 27


Resource scaling complete: L= 81


Resource scaling complete: L= 162


## 7. Controlled authorization microbenchmarks: B0 / B1 / B2

The same fully registered/authenticated/ML-KEM-established context and actual trusted public keys are used for all three modes **before starting the authorization timer**. Two independent signature-verification rounds (collector and EV analogue) are included in each mode. The B2 branch calls the **unaltered reference QSS reconstruction and token-issuance functions**; the classical branches construct a comparison-only token with the same *field types*, binding context, resource label, validity window and an opaque 32-byte digest. Baselines are not complete independent V2G systems, and do not reproduce registration, CS validation or QSS resource preparation during the timed portion.

Do not interpret the single-signer B0 microbenchmark as threshold authorization. Its trust model is weaker; B1 has two distinct signers but no quantum resource. B2 remains signature-attested, not a cryptographically verified VQSS proof.

In [7]:
def context_ready(pair,trial):
    # Setup excluded from the microbenchmark for all three comparison modes.
    p=QSV2G(crypto=pqc,seed=SEED+trial,L=162)
    p.register()
    p.authenticate()
    p.establish_session()
    p.authorization_context(op='CHARGE' if trial%2==0 else 'DISCHARGE')
    p.authority_verify(approvals=('A','B','C'))
    assert p.stage=='APPROVED'
    return p

def check_twice(p, statement, signatures):
    msg=encode(*statement)
    for _ in range(2):
        assert all(p.pqc.verify(p.sigkeys[a][0],msg,sig) for a,sig in signatures.items())

def authorization_mode(p,pair,mode):
    t0=time.perf_counter_ns()
    if mode=='B2_QSV2G_QSS_plus_2_signers':
        p.threshold_reconstruct(pair=pair)
        p.issue_token()  # internally verifies two signatures once
        statement=p.token['statement'];signatures=p.token['signatures']
        # EV-equivalent second verification round; full EV response done in end-to-end test.
        msg=encode(*statement)
        assert all(p.pqc.verify(p.sigkeys[a][0],msg,sig) for a,sig in signatures.items())
        assert p.resource.state=='CONSUMED'
    else:
        signers=(pair[0],) if mode=='B0_single_signer' else tuple(pair)
        h=H('QSV2G-CONTEXT-ONLY-APPROVAL',p.ac,p.sid,p.resource.nu,p.vld)
        statement=('QSV2G-AUTHZ',p.ac,p.sid,p.resource.nu,signers[0],signers[1] if len(signers)>1 else 'NONE',h,p.vld,'ACCEPT')
        msg=encode(*statement)
        signatures={a:p.pqc.sign(p.sigkeys[a][1],msg) for a in signers}
        check_twice(p,statement,signatures)
    ms=(time.perf_counter_ns()-t0)/1e6
    data=canon({'statement':statement,'signatures':signatures})
    return dict(mode=mode,authorization_ms=ms,signer_count=len(signatures),signatures_bytes=sum(map(len,signatures.values())),
     token_serialized_bytes=len(data),qss_reconstruction_included=(mode=='B2_QSV2G_QSS_plus_2_signers'),
     token_scope='benchmark comparison, not instrumented network packet')

base_rows=[]
MODES=['B0_single_signer','B1_two_classical_signers','B2_QSV2G_QSS_plus_2_signers']
for pair in PAIRS:
    for k in range(WARMUPS+REPEATS):
        prepared=context_ready(pair,1000*len(base_rows)+k)
        # Rotate non-QSS order to limit systematic warm-cache effects; B2 can run only once.
        order=MODES if (k%2==0) else [MODES[1],MODES[0],MODES[2]]
        for mode in order:
            result=authorization_mode(prepared,pair,mode)
            if k>=WARMUPS:
                base_rows.append(dict(pair=''.join(pair),trial=k-WARMUPS,backend=pqc.kind,**result))
    report('Authorization microbenchmarks complete:', ''.join(pair))
baseline_df=pd.DataFrame(base_rows)
assert len(baseline_df)==len(PAIRS)*REPEATS*len(MODES)
assert baseline_df.groupby(['pair','mode']).size().eq(REPEATS).all()
assert all(baseline_df[baseline_df['mode']=='B0_single_signer'].signer_count==1)
assert all(baseline_df[baseline_df['mode']!='B0_single_signer'].signer_count==2)
baseline_df.to_csv(OUT/'authorization_baselines_raw.csv',index=False)
report('Measured authorization microbenchmarks:',len(baseline_df),'with balanced pairs and modes.')

Authorization microbenchmarks complete: AB


Authorization microbenchmarks complete: AC


KeyboardInterrupt: 

## 8. Cryptographic object sizes and honest communication boundaries

In [ ]:
# These are actual sizes from the selected standardized PQC implementation, not a packet trace.
kem_pk,kem_sk=pqc.kem_keygen()
ct,ss=pqc.encaps(kem_pk)
assert ss==pqc.decaps(kem_sk,ct)
sig_pk,sig_sk=pqc.sig_keygen()
signature=pqc.sign(sig_sk,b'QSV2G-STAGE04-SIZE-VALIDATION')
assert pqc.verify(sig_pk,b'QSV2G-STAGE04-SIZE-VALIDATION',signature)
crypto_sizes=pd.DataFrame([
 {'object':'ML-KEM-768 public key','size_bytes':len(kem_pk),'scope':'actual backend output; encoding backend dependent'},
 {'object':'ML-KEM-768 secret key','size_bytes':len(kem_sk),'scope':'actual backend output; not sent as protocol message'},
 {'object':'ML-KEM-768 ciphertext','size_bytes':len(ct),'scope':'actual backend output'},
 {'object':'ML-KEM-768 shared secret','size_bytes':len(ss),'scope':'actual backend output; not sent as plaintext'},
 {'object':'ML-DSA-65 public key','size_bytes':len(sig_pk),'scope':'actual backend output; encoding backend dependent'},
 {'object':'ML-DSA-65 secret key','size_bytes':len(sig_sk),'scope':'actual backend output; not sent as protocol message'},
 {'object':'ML-DSA-65 signature','size_bytes':len(signature),'scope':'actual backend output'}])
crypto_sizes.to_csv(OUT/'pqc_object_sizes.csv',index=False)
packet_summary=packet_df.groupby(['L_trits','message']).agg(samples=('wire_bytes','size'),median_bytes=('wire_bytes','median'),min_bytes=('wire_bytes','min'),max_bytes=('wire_bytes','max')).reset_index()
packet_summary.to_csv(OUT/'instrumented_aead_packets_summary.csv',index=False)
wire_totals=packet_df.groupby(['L_trits','trial']).wire_bytes.sum().reset_index(name='selected_aead_bytes_total')
wire_totals.to_csv(OUT/'selected_aead_packet_totals.csv',index=False)
report('Selected AEAD packet totals; NOT total protocol traffic:')
report(wire_totals.groupby('L_trits').selected_aead_bytes_total.median().to_string())

## 9. Statistics, consistency checks, and legacy comparisons

In [ ]:
def descriptive(df,groupcols,valcol):
    out=[]
    for key,g in df.groupby(groupcols,sort=True):
        key=key if isinstance(key,tuple) else (key,)
        vals=g[valcol].to_numpy(dtype=float)
        out.append(dict(zip(groupcols,key),samples=len(vals),median=float(np.median(vals)),mean=float(np.mean(vals)),
         std_sample=float(np.std(vals,ddof=1)) if len(vals)>1 else float('nan'),
         p95=float(np.percentile(vals,95)),minimum=float(np.min(vals)),maximum=float(np.max(vals)),metric=valcol))
    return pd.DataFrame(out)
end_stats=descriptive(full_df,['L_trits'],'total_wall_ms')
phase_stats=descriptive(phase_df,['L_trits','phase'],'phase_elapsed_ms')
qss_stats=pd.concat([descriptive(qss_df,['L_trits'],x) for x in ['resource_preparation_ms','reconstruction_ms']],ignore_index=True)
base_stats=descriptive(baseline_df,['pair','mode'],'authorization_ms')
end_stats.to_csv(OUT/'end_to_end_statistics.csv',index=False)
phase_stats.to_csv(OUT/'phase_statistics.csv',index=False)
qss_stats.to_csv(OUT/'qss_scaling_statistics.csv',index=False)
base_stats.to_csv(OUT/'authorization_baselines_statistics.csv',index=False)
legacy_df=pd.read_csv(io.BytesIO(oldtimings))
legacy_packet_df=pd.read_csv(io.BytesIO(oldpackets))
assert len(legacy_df)==8 and 'total_wall_ms' in legacy_df.columns
# These files are preserved for reference; do not pool cross-session/platform samples.
comparisons={'previous_reported_backend':str(legacy_df['backend'].iloc[0]),'current_backend':pqc.kind,
 'previous_n':len(legacy_df),'current_n_at_L162':int((full_df.L_trits==162).sum()),
 'previous_median_ms':float(legacy_df.total_wall_ms.median()),
 'current_median_ms_L162':float(full_df.query('L_trits==162').total_wall_ms.median()),
 'note':'Descriptive only. Not pooled, not a regression claim; hosts/package versions differ.'}
(OUT/'legacy_comparison_context.json').write_text(json.dumps(comparisons,indent=2))
assert all(end_stats['samples']==REPEATS)
assert all(base_stats['samples']==REPEATS)
assert set(packet_df.message)==set(probe.wire_bytes)
report('End-to-end latency (ms) by L:')
report(end_stats[['L_trits','median','p95','mean','std_sample']].to_string(index=False))
report('Authorization comparison (medians in ms, see raw per pair):')
report(baseline_df.groupby('mode').authorization_ms.median().round(3).to_string())

## 10. Publishable candidate figures (axis units, counts, and labels explicit)

In [ ]:
plots=[]
def saveplot(name):
    plt.tight_layout();dest=OUT/name;plt.savefig(dest,dpi=190,bbox_inches='tight');plt.close();plots.append(name)
fig,ax=plt.subplots(figsize=(7.7,4.6))
ax.errorbar(end_stats.L_trits,end_stats['median'],yerr=[end_stats['median']-end_stats['minimum'],end_stats['maximum']-end_stats['median']],fmt='o-',capsize=4)
ax.set(xlabel='QSS trits per dealer L',ylabel='End-to-end measured wall time (ms)',title='QSV2G reference workflow — median and observed min/max')
ax.set_xticks(L_VALUES);ax.grid(alpha=0.25)
ax.text(0.03,0.03,f'{REPEATS} measured runs per L; one backend: {pqc.kind}',transform=ax.transAxes,fontsize=8)
saveplot('figure_end_to_end_scaling.png')

fig,ax=plt.subplots(figsize=(7.7,4.6))
for metric,label in [('resource_preparation_ms','QSS resource preparation'),('reconstruction_ms','QSS reconstruction')]:
    summary=descriptive(qss_df,['L_trits'],metric)
    ax.plot(summary.L_trits,summary['median'],'o-',label=label)
ax.set(xlabel='QSS trits per dealer L',ylabel='Median local CPU wall time (ms)',title='Ideal resource preparation and threshold reconstruction')
ax.set_xticks(L_VALUES);ax.legend();ax.grid(alpha=0.25)
saveplot('figure_qss_scaling.png')

fig,ax=plt.subplots(figsize=(8.2,4.5))
mode_median=baseline_df.groupby('mode').authorization_ms.median().reindex(MODES)
ax.bar(['B0: 1 signature','B1: 2 signatures','B2: 2 signatures + QSS'],mode_median.values)
ax.set(ylabel='Median authorization microbenchmark time (ms)',title='Same pre-authorized context; crypto & QSS only')
ax.tick_params(axis='x',labelsize=8);ax.grid(axis='y',alpha=0.25)
saveplot('figure_authorization_baselines.png')

fig,ax=plt.subplots(figsize=(8.0,4.4))
by_packet=packet_df[packet_df.L_trits==162].groupby('message').wire_bytes.median().sort_values()
ax.barh(by_packet.index,by_packet.values)
ax.set(xlabel='Observed AEAD packet size (bytes)',title='Measured selected classical AEAD packets, L=162')
ax.grid(axis='x',alpha=0.25)
saveplot('figure_instrumented_aead_packets.png')

fig,ax=plt.subplots(figsize=(7.7,4.4))
ax.plot(L_VALUES,[6*x for x in L_VALUES],'o-',label='QSS remote qutrit transmissions (analytical)')
ax.plot(L_VALUES,[9*x for x in L_VALUES],'s--',label='QSS stored qutrit subsystems (analytical)')
ax.set(xlabel='QSS trits per dealer L',ylabel='Count of qutrit subsystems',title='Model-based quantum resource accounting (not hardware measurements)')
ax.legend();ax.set_xticks(L_VALUES);ax.grid(alpha=0.25)
saveplot('figure_analytical_quantum_resources.png')
report('Generated figures:',plots)

## 11. Environment, provenance and run completion checks

Interpret the output cautiously: microbenchmark timing measures a particular machine and provider, and the `authorization_token` phase includes ML-DSA signing and signature verification. The B2 timing includes QSS reconstruction **but excludes resource setup**, which is reported separately. The total encrypted packet bytes shown here are *instrumented selected messages only*. No throughput, battery/cryptographic energy, full-network delivery, or QSS noise effect on complete transaction completion was measured.

In [ ]:
env={
 'python':sys.version,'platform':platform.platform(),'os':os.name,
 'numpy':np.__version__,'pandas':pd.__version__,'matplotlib':matplotlib.__version__,
 'cryptography':version_or_none('cryptography'),'pqcrypto':version_or_none('pqcrypto'),
 'backend':pqc.kind,'openssl_version':subprocess.run(['openssl','version'],capture_output=True,text=True).stdout.strip() if shutil.which('openssl') else None,
 'kaggle_path_detected':IS_KAGGLE,'note':'Environment metadata is reported by this execution, not cryptographic attestation.'}
(OUT/'environment.json').write_text(json.dumps(env,indent=2))
(OUT/'source_hashes.json').write_text(json.dumps({
 'core_sha256':CORE_SHA,'section_iv_sha256':TEX_SHA,
 'previous_reported_timing_csv_sha256':OLD_TIMINGS_SHA,
 'previous_aead_packet_csv_sha256':OLD_PACKETS_SHA},indent=2))
summary={
 'study':'QSV2G Stage04 software performance and constructed baseline microbenchmarks',
 'status':'PASS_WITH_RESEARCH_LIMITATIONS',
 'full_workflows':len(full_df),'full_runs_by_L':{str(k):int(v) for k,v in full_df.groupby('L_trits').size().items()},
 'resource_runs':len(qss_df),'authorization_microbenchmark_runs':len(baseline_df),
 'baseline_runs_per_pair_mode':REPEATS,'warmups_per_condition':WARMUPS,
 'pq_backend':pqc.kind,'source_hashes':{'core':CORE_SHA,'section_iv':TEX_SHA},
 'median_end_to_end_ms_by_L':{str(r.L_trits):r.median for r in end_stats.itertuples()},
 'median_authorization_ms_by_mode':{str(k):float(v) for k,v in baseline_df.groupby('mode').authorization_ms.median().items()},
 'measured_aead_packet_coverage':sorted(packet_df.message.unique().tolist()),
 'claims_not_tested':['real quantum network delay or fidelity','physical V2G','cryptographic energy use',
  'formal VQSS security','complete classical baseline architecture','quantum advantage',
  'Kaggle immutable run provenance without run URL']}
(OUT/'stage04_summary.json').write_text(json.dumps(summary,indent=2))
(OUT/'run_log.txt').write_text('\n'.join(LOG)+'\n')
(OUT/'README_RESULTS.md').write_text("""# Stage 04 results (QSV2G)\n\n- `end_to_end_raw.csv`: unmodified QSV2G `run_demo()` end-to-end wall clock; warmups omitted.\n- `phase_latency_raw.csv`: phase durations from the original reference log; do not assume phase-duration sums equal end-to-end timing exactly.\n- `qss_resource_scaling_raw.csv`: separate preparation/reconstruction wall time and analytical remote-transmission/subsystem counts.\n- `authorization_baselines_raw.csv`: one-signer B0, two-classical-signer B1, original-QSS B2 **post-context microbenchmarks**. Not full baseline V2G protocols.\n- `instrumented_aead_packets_raw.csv`: subset of five AEAD packet types, not complete classical V2G communication.\n- `pqc_object_sizes.csv`: genuine provider sizes; PEM encoding changes byte lengths when OpenSSL is used.\n- `*_statistics.csv`, `figure_*.png`: computed summaries and candidate figures.\n- `legacy_*.csv`: original reported eight-run benchmark and AEAD packet data retained for provenance, not pooled with new measurements.\n- `environment.json`, `source_hashes.json`, `manifest.json`, `run_log.txt`: reproducibility and validation metadata.\n\n**Important**: No quantum physical channel, real grid meter, energy estimate for cryptography, or end-to-end comparison of independently implemented alternative trust architectures. Source and protocol hashes refer to the verified historical Section IV reference, which must be reconciled with any newer edited paper version.\n""")
assert summary['status']=='PASS_WITH_RESEARCH_LIMITATIONS' and summary['full_workflows']==120
assert summary['resource_runs']==120 and summary['authorization_microbenchmark_runs']==270
assert (full_df.total_wall_ms>0).all() and (baseline_df.authorization_ms>0).all()
report('STAGE RESULT: PASS_WITH_RESEARCH_LIMITATIONS')
report('Full workflows:',len(full_df),'QSS resource trials:',len(qss_df),'baseline trials:',len(baseline_df))

## 12. Export ZIP with SHA-256 integrity manifest

In [ ]:
# Manifest lists every output other than itself to avoid a self-referential digest.
contents=sorted([f for f in OUT.iterdir() if f.is_file() and f.name!='manifest.json'])
manifest={'stage':'04','files':[{ 'path':f.name,'bytes':f.stat().st_size,'sha256':sha(f.read_bytes())} for f in contents],
 'source_code_sha256':CORE_SHA,'protocol_section_sha256':TEX_SHA,
 'generated_at_utc':datetime.now(timezone.utc).isoformat(),
 'manifest_note':'Hashes verify artifact integrity, not authenticity of cloud provenance or security proof.'}
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
    for f in sorted(OUT.iterdir()):
        if f.is_file():zf.write(f,arcname=f.name)
with zipfile.ZipFile(ZIP_PATH) as zf:
    assert zf.testzip() is None
    listing=set(zf.namelist())
    assert listing==set(x.name for x in OUT.iterdir() if x.is_file())
    m=json.loads(zf.read('manifest.json'))
    assert all(len(zf.read(x['path']))==x['bytes'] and sha(zf.read(x['path']))==x['sha256'] for x in m['files'])
report('ZIP OUTPUT:',ZIP_PATH,'size bytes:',ZIP_PATH.stat().st_size)
report('VERIFIED ARTIFACTS:',len(manifest['files']),'plus manifest.json')
report('STAGE RESULT: PASS_WITH_RESEARCH_LIMITATIONS')